# 01: TMDB Overview and Basic Cleaning of the Dataset

## Setup

### Imports & configuration

In [ ]:
%matplotlib inline

import sys
import warnings
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config
from src.eda.helper import (
    parse_list_field,
    text_field_report,
)
from src.eda.loaders import load_tmdb
from src.eda.report import log_findings, savefig

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PROCESSED_DIR = config.PROCESSED_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# This notebook's own figures subdirectory (src/eda/report.py's per-notebook
# convention) -- bound into savefig once here so every savefig(fig, filename)
# call below lands in reports/figures/01/ without repeating figures_dir.
FIGURES_DIR = config.FIGURES_DIR / "01"
savefig = partial(savefig, figures_dir=FIGURES_DIR)

### Load dataset

In [ ]:
tmdb_raw = load_tmdb()

print("TMDB raw shape:", tmdb_raw.shape)

## 1. Structural overview

### Data provenance

In [ ]:
display(Markdown(f"""
**Dataset provenance** (pulled from `src/eda/config.py`):

- TMDB download date: `{config.TMDB_DOWNLOAD_DATE}`
"""))

### Shape, dtypes & column list

In [ ]:
tmdb_shape_raw = tmdb_raw.shape
print("TMDB shape:", tmdb_shape_raw)

In [ ]:
tmdb_raw.dtypes

### Memory usage

In [ ]:
tmdb_mem_mb = tmdb_raw.memory_usage(deep=True).sum() / 1e6
print(f"TMDB memory usage (deep): {tmdb_mem_mb:,.1f} MB")

### Random sample inspection

In [ ]:
tmdb_raw.sample(10, random_state=config.RANDOM_SEED)

### Missingness of raw data

In [ ]:
tmdb_missing = pd.DataFrame(
    {
        "n_missing": tmdb_raw.isnull().sum(),
        "pct_missing": tmdb_raw.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
tmdb_missing

In [ ]:
msno.matrix(tmdb_raw.sample(min(len(tmdb_raw), 5000), random_state=config.RANDOM_SEED))
plt.title("TMDB missingness pattern (sample)")
fig = plt.gcf()
savefig(fig, "01_tmdb_missingness_matrix_rawdata.png")
plt.show()

### Duplicate `id` check

In [ ]:
tmdb_dupe_ids = int(tmdb_raw["id"].duplicated().sum())
print(f"TMDB duplicate ids: {tmdb_dupe_ids}")

In [ ]:
# Do duplicate-id rows actually agree on core identity fields, or do they
# genuinely conflict (which would mean the same id was reused across two
# different movies, not just re-entered with more fields filled in)?
key_cols = ["imdb_id", "title", "release_date", "vote_average"]
tmdb_dupe_id_rows = tmdb_raw[tmdb_raw["id"].duplicated(keep=False)]
n_distinct_non_null = tmdb_dupe_id_rows.groupby("id")[key_cols].agg(lambda s: s.dropna().nunique())
conflicting_ids = n_distinct_non_null[(n_distinct_non_null > 1).any(axis=1)]
n_dupe_id_groups = tmdb_dupe_id_rows["id"].nunique()
print(f"{len(conflicting_ids)} of {n_dupe_id_groups} duplicate-id groups "
      f"have conflicting (not just missing-vs-present) values in {key_cols}")

tmdb_dupe_id_rows[tmdb_dupe_id_rows["id"].isin(conflicting_ids.index)][["id"] + key_cols].sort_values("id").head()

### Duplicate `imdb_id` check

In [ ]:
tmdb_imdb_id_present = tmdb_raw["imdb_id"].dropna()
tmdb_imdb_id_present = tmdb_imdb_id_present[tmdb_imdb_id_present != "tt0000000"]
tmdb_dupe_imdb_id_mask = tmdb_imdb_id_present.duplicated(keep=False)
tmdb_dupe_imdb_ids = int(tmdb_imdb_id_present.duplicated().sum())
print(f"TMDB duplicate imdb_ids (excluding missing/placeholder values): {tmdb_dupe_imdb_ids} "
      f"out of {len(tmdb_imdb_id_present)} non-null, non-placeholder values")

tmdb_dupe_imdb_id_rows = tmdb_raw.loc[
    tmdb_imdb_id_present[tmdb_dupe_imdb_id_mask].index,
    ["id", "imdb_id", "title", "release_date"],
].sort_values("imdb_id")
tmdb_dupe_imdb_id_rows.head(20)

### `vote_average` / `vote_count` zero combinations

In [ ]:
vote_avg_zero = tmdb_raw["vote_average"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((vote_avg_zero & vote_count_zero).sum())
n_avg_nonzero_count_zero = int((~vote_avg_zero & vote_count_zero).sum())
n_avg_zero_count_nonzero = int((vote_avg_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~vote_avg_zero & ~vote_count_zero).sum())

print(f"vote_average==0 and vote_count==0:    {n_both_zero}")
print(f"vote_average!=0 and vote_count==0:    {n_avg_nonzero_count_zero}")
print(f"vote_average==0 and vote_count!=0:    {n_avg_zero_count_nonzero}")
print(f"vote_average!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                                {len(tmdb_raw)}")

DECISION: vote_count==0 means zero votes were cast, so vote_average cannot be anything but 0 in that case -- an average requires at least one value to average. The 582 rows above with vote_count==0 but vote_average!=0 are therefore logically impossible, not just noisy: corrupted/erroneous records.

Conversely, vote_count!=0 means votes were cast, so vote_average should not be zero -- vote_average==0 despite votes being cast is therefore also a corrupted/erroneous record.

### `popularity` / `vote_count` zero combinations

In [ ]:
popularity_zero = tmdb_raw["popularity"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((popularity_zero & vote_count_zero).sum())
n_pop_nonzero_count_zero = int((~popularity_zero & vote_count_zero).sum())
n_pop_zero_count_nonzero = int((popularity_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~popularity_zero & ~vote_count_zero).sum())

print(f"popularity==0 and vote_count==0:    {n_both_zero}")
print(f"popularity!=0 and vote_count==0:    {n_pop_nonzero_count_zero}")
print(f"popularity==0 and vote_count!=0:    {n_pop_zero_count_nonzero}")
print(f"popularity!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                              {len(tmdb_raw)}")


### Null or empty strings in `title`

In [ ]:
tmdb_title_nulls, tmdb_title_empty = text_field_report(tmdb_raw, "title", "TMDB title")

### Erroneous `runtime`, `budget`, and `revenue` values

In [ ]:
impossible_flags = []
if "runtime" in tmdb_raw.columns:
    impossible_flags.append(("runtime == 0", int((tmdb_raw["runtime"] == 0).sum())))
    impossible_flags.append(("runtime < 0", int((tmdb_raw["runtime"] < 0).sum())))
if "revenue" in tmdb_raw.columns:
    impossible_flags.append(("revenue < 0", int((tmdb_raw["revenue"] < 0).sum())))
if "budget" in tmdb_raw.columns:
    impossible_flags.append(("budget < 0", int((tmdb_raw["budget"] < 0).sum())))

pd.DataFrame(impossible_flags, columns=["check", "n_rows"])

## 2. Cleaning

In [ ]:
tmdb_clean = tmdb_raw.copy()

### Drop URL features

In [ ]:
url_features = ['homepage', 'poster_path', 'backdrop_path']
tmdb_clean = tmdb_clean.drop(columns=url_features)

URL features are not addressed in this dissertation.

### Filter to `status == 'Released'`

This filter is necessary because `vote_average` is only meaningful once a movie's status is `'Released'` -- if `vote_average > 0` while `status` is not `'Released'`, those records are erroneous. Some features, like `imdb_id`, are also only assigned after a movie is released.

In [ ]:
status_counts = tmdb_clean["status"].value_counts(dropna=False)
status_pct = (status_counts / len(tmdb_clean) * 100).round(2)
pd.DataFrame({"count": status_counts, "pct": status_pct})

In [ ]:
# Verify the claim behind the filter across every column, not just the
# handful expected to be post-release-only.
released_mask = tmdb_clean["status"] == "Released"
missingness_by_status = pd.DataFrame({
    "% missing (Released)": tmdb_clean.loc[released_mask].isnull().mean() * 100,
    "% missing (non-Released)": tmdb_clean.loc[~released_mask].isnull().mean() * 100,
})
missingness_by_status["gap"] = missingness_by_status["% missing (non-Released)"] - missingness_by_status["% missing (Released)"]
missingness_by_status.round(2).sort_values("gap", ascending=False)

In [ ]:
tmdb_clean = tmdb_clean.copy()
shape_before_status_filter = tmdb_clean.shape
tmdb_clean = tmdb_clean[tmdb_clean["status"] == "Released"].copy()
tmdb_shape_after_status_filter = tmdb_clean.shape
print(f"TMDB shape before status filter: {shape_before_status_filter}")
print(f"TMDB shape after status filter: {tmdb_shape_after_status_filter}")

### Filter to `adult == False`

Personal choice of not dealing with adult-themed movies.
Also, this filter is not 100% correct, as the values of this will persist

In [ ]:
tmdb_shape_before_adult_filter = tmdb_clean.shape
tmdb_clean = tmdb_clean[tmdb_clean["adult"] == False]
print(f"Shape: {tmdb_shape_before_adult_filter} -> {tmdb_clean.shape}")

### Drop the filter features

After filtering, each of these features has only one remaining value: `status` is always `'Released'` and `adult` is always `False`. Keeping them would just be extra features with no information to learn from.

In [ ]:
filtered_features = ["status", "adult"]
tmdb_clean = tmdb_clean.drop(columns=filtered_features)

### Recode `budget`/`revenue` zeros as NaN

Realistically, `budget > 0` and `revenue > 0`.

In [ ]:
zero_recode_report = {}
for col in ["budget", "revenue"]:
    pct_zero = float((tmdb_clean[col] == 0).mean() * 100)
    zero_recode_report[col] = pct_zero
    tmdb_clean[col] = tmdb_clean[col].replace(0, np.nan)
    print(f"{col}: {pct_zero:.1f}% of rows were 0 -> recoded as NaN")

### Recode `runtime` zeros as NaN

The runtime of a movie cannot be zero.

In [ ]:
pct_zero_runtime = float((tmdb_clean["runtime"] == 0).mean() * 100)
tmdb_clean["runtime"] = tmdb_clean["runtime"].replace(0, np.nan)
print(f"runtime: {pct_zero_runtime:.1f}% of rows were 0 -> recoded as NaN")

### Recode negative `runtime`/`revenue` values as NaN

In [ ]:
tmdb_neg_recode_report = {}
for col in ["runtime", "revenue"]:
    pct_negative = float((tmdb_clean[col] < 0).mean() * 100)
    tmdb_neg_recode_report[col] = pct_negative
    tmdb_clean[col] = tmdb_clean[col].where(tmdb_clean[col] >= 0, np.nan)
    print(f"{col}: {pct_negative:.4f}% of rows were negative -> recoded as NaN")

### Drop `runtime` > 200

The 99th-percentile check above confirms `runtime`'s extreme skew is a thin tail: drops rows with `runtime > 200`.

In [ ]:
print(f"99th percentile of runtime: {tmdb_clean['runtime'].quantile(0.99):.1f} minutes")

In [ ]:
shape_before_runtime_cap = tmdb_clean.shape
tmdb_clean = tmdb_clean[tmdb_clean["runtime"] <= 200]
pct_dropped_runtime = (
    (shape_before_runtime_cap[0] - tmdb_clean.shape[0])
    / shape_before_runtime_cap[0] * 100
)
print(
    f"tmdb_clean: {shape_before_runtime_cap} -> {tmdb_clean.shape} "
    f"({pct_dropped_runtime:.2f}% dropped)"
)

### Drop missing `runtime` rows

In [ ]:
tmdb_shape_before_runtime_drop = tmdb_clean.shape
tmdb_clean = tmdb_clean[tmdb_clean["runtime"].notna()]
print(f"Dropped {tmdb_shape_before_runtime_drop[0] - tmdb_clean.shape[0]} rows with runtime == NaN")
print(f"Shape: {tmdb_shape_before_runtime_drop} -> {tmdb_clean.shape}")

### Drop duplicate `id` rows

In [ ]:
tmdb_shape_before_dedup = tmdb_clean.shape
tmdb_clean = (
    tmdb_clean.assign(_null_count=tmdb_clean.isnull().sum(axis=1))
    .sort_values("_null_count")
    .drop_duplicates(subset="id", keep="first")
    .drop(columns="_null_count")
    .sort_index()
)
tmdb_shape_after_dedup = tmdb_clean.shape
tmdb_n_dupe_ids_dropped = tmdb_shape_before_dedup[0] - tmdb_shape_after_dedup[0]
print(f"TMDB: dropped {tmdb_n_dupe_ids_dropped} duplicate-id rows (kept the most complete row per id)")
print(f"Shape before dedup: {tmdb_shape_before_dedup}, after: {tmdb_shape_after_dedup}")

### Validate `imdb_id` duplicates before dropping them

Duplicate `imdb_id` rows in this dataset are largely localized/dubbed re-entries of the same movie (see the raw duplicate-`imdb_id` sample in Section 1), so `title`/`original_title`/`overview`/`tagline` are expected to differ and are excluded here. `vote_average`/`vote_count` are also excluded -- they are per-TMDB-entry counters, not movie-level facts, so they're expected to differ across separate listings of the same film. This checks whether the facts that should NOT depend on either of those -- `budget`, `revenue`, `runtime`, `release_date` -- have a genuine conflict (two non-null, disagreeing values) within each duplicate-`imdb_id` group, to confirm the "keep the most complete row" dedup below is safe rather than merging genuinely different records.

In [ ]:
CORE_FACT_COLS = ["budget", "revenue", "runtime", "release_date"]

dupe_imdb_mask = tmdb_clean["imdb_id"].duplicated(keep=False) & tmdb_clean["imdb_id"].notna()
tmdb_dupe_imdb_groups = tmdb_clean[dupe_imdb_mask]

def has_conflict(group):
    # A real conflict: two non-null values disagree on some column.
    # A row with a missing value doesn't count -- that's incompleteness, not conflict.
    return any(group[col].dropna().nunique() > 1 for col in CORE_FACT_COLS)

conflicting_imdb_groups = tmdb_dupe_imdb_groups.groupby("imdb_id").filter(has_conflict)
n_conflicting_groups = conflicting_imdb_groups["imdb_id"].nunique()
n_total_dupe_groups = tmdb_dupe_imdb_groups["imdb_id"].nunique()
print(f"{n_conflicting_groups} of {n_total_dupe_groups} duplicate-imdb_id groups have a genuine "
      f"conflict (two non-null, disagreeing values) in {CORE_FACT_COLS}")

### Drop duplicate `imdb_id` rows

In [ ]:
tmdb_shape_before_imdb_dedup = tmdb_clean.shape

imdb_id_present = tmdb_clean["imdb_id"].notna()
tmdb_imdb_dupes = tmdb_clean[imdb_id_present]
tmdb_no_imdb_id = tmdb_clean[~imdb_id_present]

tmdb_imdb_dupes = (
    tmdb_imdb_dupes.assign(_null_count=tmdb_imdb_dupes.isnull().sum(axis=1))
    .sort_values(["_null_count", "id"])
    .drop_duplicates(subset="imdb_id", keep="first")
    .drop(columns="_null_count")
)

tmdb_clean = pd.concat([tmdb_imdb_dupes, tmdb_no_imdb_id]).sort_index()

tmdb_shape_after_imdb_dedup = tmdb_clean.shape
tmdb_n_dupe_imdb_ids_dropped = tmdb_shape_before_imdb_dedup[0] - tmdb_shape_after_imdb_dedup[0]
print(f"TMDB: dropped {tmdb_n_dupe_imdb_ids_dropped} duplicate-imdb_id rows (kept the most complete row per imdb_id, tie-broken by lowest id)")
print(f"Shape before imdb_id dedup: {tmdb_shape_before_imdb_dedup}, after: {tmdb_shape_after_imdb_dedup}")

### Drop rows with missing `title`

In [ ]:
tmdb_shape_before_title_drop = tmdb_clean.shape
tmdb_clean = tmdb_clean[tmdb_clean["title"].notna()]
print(f"Dropped {tmdb_shape_before_title_drop[0] - tmdb_clean.shape[0]} rows with title == NaN")
print(f"Shape: {tmdb_shape_before_title_drop} -> {tmdb_clean.shape}")

### Remove inconsistent `vote_average`/`vote_count` rows

In [ ]:
tmdb_shape_before_vote_fix = tmdb_clean.shape
bad_vote_mask = (
    ((tmdb_clean["vote_average"] != 0) & (tmdb_clean["vote_count"] == 0))
    | ((tmdb_clean["vote_average"] == 0) & (tmdb_clean["vote_count"] != 0))
)
tmdb_clean = tmdb_clean[~bad_vote_mask]
print(f"Dropped {int(bad_vote_mask.sum())} rows with inconsistent vote_average/vote_count")
print(f"Shape: {tmdb_shape_before_vote_fix} -> {tmdb_clean.shape}")

### Convert list-valued string features to list type

Covers TMDB `genres`, `keywords`, `production_companies`,
`production_countries`, and `spoken_languages`. TMDB stores these as
comma-separated plain strings rather than JSON, so `parse_list_field()`
(defined in Setup) tries `json.loads` / `ast.literal_eval` first and
falls back to a comma split otherwise.

In [ ]:
list_features = ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"]

#### Parsing verification

In [ ]:
for col, df_ in [
    ("genres", tmdb_clean),
    ("keywords", tmdb_clean),
    ("production_companies", tmdb_clean),
    ("production_countries", tmdb_clean),
    ("spoken_languages", tmdb_clean),
]:
    sample_val = df_[col].dropna().iloc[0]
    print(f"--- {col} ---")
    print("raw:", sample_val)
    print("parsed:", parse_list_field(sample_val))
    print()

#### Apply parsing

In [ ]:
for col in list_features:
    tmdb_clean[col] = tmdb_clean[col].apply(parse_list_field)


# Replace empty lists with NaN
for col in list_features:
    tmdb_clean[col] = tmdb_clean[col].apply(lambda lst: lst if lst else np.nan)

### Missingness after cleaning

In [ ]:
tmdb_missing = pd.DataFrame(
    {
        "n_missing": tmdb_clean.isnull().sum(),
        "pct_missing": tmdb_clean.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
tmdb_missing

In [ ]:
msno.matrix(tmdb_clean.sample(min(len(tmdb_clean), 5000), random_state=config.RANDOM_SEED))
plt.title("TMDB missingness pattern (sample)")
fig = plt.gcf()
savefig(fig, "01_tmdb_missingness_matrix_cleandata.png")
plt.show()

### View dataset after cleaning

In [ ]:
tmdb_clean.sample(10, random_state=config.RANDOM_SEED)

## 3. Candidate outcome variable distribution

`vote_average` is chosen as the target label. This is needed for the supervised learning paradigm.

### `vote_average` distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_clean["vote_average"].dropna().hist(bins=40, ax=ax)
ax.set_title("TMDB: vote_average distribution")
ax.set_xlabel("vote_average")
ax.set_ylabel("count")
savefig(fig, "01_tmdb_vote_average_hist.png")
plt.show()

### `vote_count` threshold counts

In [ ]:
for threshold in [1, 5, 10, 20, 50]:
    print(f"vote_count >= {threshold}: {(tmdb_clean['vote_count'] >= threshold).sum():,} rows")

## 4. Save the cleaned file

In [ ]:
tmdb_clean_path = PROCESSED_DIR / "tmdb_clean.parquet"

tmdb_clean.to_parquet(tmdb_clean_path, index=False)

print(f"Saved: {tmdb_clean_path} ({tmdb_clean.shape})")

## 5. Log summary

In [ ]:
findings = {
    "TMDB rows (raw -> cleaned)": f"{tmdb_shape_raw[0]:,} -> {tmdb_clean.shape[0]:,}",
    "TMDB columns (raw -> cleaned)": f"{tmdb_shape_raw[1]} -> {tmdb_clean.shape[1]}",
    "Raw memory footprint (deep)": f"{tmdb_mem_mb:,.1f} MB",
    "Dropped: status != 'Released'": f"{shape_before_status_filter[0] - tmdb_shape_after_status_filter[0]:,} rows",
    "Dropped: adult == True": f"{tmdb_shape_before_adult_filter[0] - tmdb_shape_before_dedup[0]:,} rows",
    "Recoded budget/revenue zeros as NaN": f"{zero_recode_report['budget']:.1f}% / {zero_recode_report['revenue']:.1f}% of rows",
    "Recoded runtime zeros as NaN": f"{pct_zero_runtime:.1f}% of rows",
    "Recoded negative runtime/revenue as NaN": f"{tmdb_neg_recode_report['runtime']:.4f}% / {tmdb_neg_recode_report['revenue']:.4f}% of rows",
    "Dropped: runtime > 200 minutes rows": f"{pct_dropped_runtime:.2f}% of rows dropped (runtime > 200)",
    "Dropped: missing runtime rows": f"{tmdb_shape_before_runtime_drop[0] - tmdb_shape_before_dedup[0]:,}",
    "Dropped: duplicate id rows": f"{tmdb_n_dupe_ids_dropped:,} (kept most complete row per id)",
    "Dropped: duplicate imdb_id rows": f"{tmdb_n_dupe_imdb_ids_dropped:,} (kept most complete row per imdb_id, tie-broken by lowest id)",
    "Dropped: missing title rows": f"{tmdb_shape_before_title_drop[0] - tmdb_shape_before_vote_fix[0]:,}",
    "Dropped: inconsistent vote_average/vote_count rows": f"{int(bad_vote_mask.sum()):,}",
    "Final missingness (top 3 by %)": tmdb_missing.head(3)["pct_missing"].round(2).to_dict(),
    "vote_count >= 1 / >= 5 / >= 10": (
        f"{(tmdb_clean['vote_count'] >= 1).sum():,} / "
        f"{(tmdb_clean['vote_count'] >= 5).sum():,} / "
        f"{(tmdb_clean['vote_count'] >= 10).sum():,} rows "
        "(vote_count filter deferred to modeling stage, not yet applied)"
    ),
    "Saved to": str(tmdb_clean_path),
}

log_findings(
    title="01 Overview & Cleaning: TMDB -- summary",
    findings=findings,
    figures=[
        "reports/figures/01/01_tmdb_missingness_matrix_cleandata.png",
        "reports/figures/01/01_tmdb_vote_average_hist.png",
    ],
)
print("Logged summary to reports/eda_log.md")